In [1]:
# =========================
# Libraries
# =========================

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [ ]:
pad=''

df=pd.read_csv(pad)

In [ ]:



# ============================================================
# CONFIGURATIE
# Pas alleen dit blok aan om andere jaren te gebruiken
# ============================================================

TRAIN_JAREN = range(2018, 2024)      # jaren waarmee het model wordt getraind, dus 2018 t/m 2023
TEST_JAREN = [2024, 2025]            # jaren waarvoor echte data bestaat en foutmarges worden berekend
PROGNOSE_JAREN = [2026]              # jaren waarvoor nog geen echte data bestaat
INSTROOM_AANNAME_JAAR = 2025         # instroom van dit jaar gebruiken voor prognosejaren zonder echte instroomdata

MODELLEN_GEBRUIKEN = [
    "regressie",
    "exponential_smoothing",
    "stationair"
]


# =========================
# Toestanden
# =========================

statussen = [
    'Niveau1_jaar_1_nieuw', 'Niveau1_jaar_1', 'Niveau1_jaar_2', 'Niveau1_jaar_>2',
    'Niveau2_jaar_1_nieuw', 'Niveau2_jaar_1', 'Niveau2_jaar_2', 'Niveau2_jaar_>2',
    'Niveau3_jaar_1_nieuw', 'Niveau3_jaar_1', 'Niveau3_jaar_2', 'Niveau3_jaar_3', 'Niveau3_jaar_>3',
    'Niveau4_jaar_1_nieuw', 'Niveau4_jaar_1', 'Niveau4_jaar_2', 'Niveau4_jaar_3',
    'Niveau4_jaar_4', 'Niveau4_jaar_5', 'Niveau4_jaar_>5',
    'uitstroom'
]

toestanden = statussen

nieuwe_statussen = [
    'Niveau1_jaar_1_nieuw',
    'Niveau2_jaar_1_nieuw',
    'Niveau3_jaar_1_nieuw',
    'Niveau4_jaar_1_nieuw'
]

doorstroom_statussen = [
    s for s in toestanden
    if s not in nieuwe_statussen and s != 'uitstroom'
]


# =========================
# Validatie
# =========================

def valideer_input(df):
    verplichte_kolommen = {
        'Studiejaar',
        'huidige_status',
        'volgende_status',
        'telling'
    }

    ontbrekend = verplichte_kolommen - set(df.columns)
    if ontbrekend:
        raise ValueError(f"Ontbrekende kolommen in df: {ontbrekend}")

    onbekende_huidige_statussen = set(df['huidige_status'].dropna()) - set(toestanden)
    onbekende_volgende_statussen = set(df['volgende_status'].dropna()) - set(toestanden)

    if onbekende_huidige_statussen:
        print("Waarschuwing: onbekende huidige_statussen gevonden:")
        print(sorted(onbekende_huidige_statussen))

    if onbekende_volgende_statussen:
        print("Waarschuwing: onbekende volgende_statussen gevonden:")
        print(sorted(onbekende_volgende_statussen))


# =========================
# Jaarlijkse overgangsmatrices
# =========================

def bouw_jaar_matrices(df, jaren):
    matrices = {}

    for jaar in jaren:
        jaar_df = df[df['Studiejaar'] == jaar]

        mat_counts = pd.crosstab(
            jaar_df['huidige_status'],
            jaar_df['volgende_status'],
            values=jaar_df['telling'],
            aggfunc='sum'
        ).fillna(0)

        mat_counts = mat_counts.reindex(
            index=toestanden,
            columns=toestanden,
            fill_value=0
        )

        mat = mat_counts.div(mat_counts.sum(axis=1), axis=0).fillna(0)

        # Uitstroom is absorberend
        mat.loc['uitstroom', :] = 0
        mat.loc['uitstroom', 'uitstroom'] = 1

        matrices[jaar] = mat

    return matrices


# =========================
# Stationaire Markov matrix
# =========================

def stationaire_voorspelling(matrices, jaren=None):
    if jaren is None:
        jaren = sorted(matrices.keys())

    if len(jaren) == 0:
        raise ValueError("Geen jaren beschikbaar voor stationaire voorspelling.")

    P = sum(matrices[jaar] for jaar in jaren) / len(jaren)

    P.loc['uitstroom', :] = 0
    P.loc['uitstroom', 'uitstroom'] = 1

    return P.div(P.sum(axis=1), axis=0).fillna(0)


# =========================
# Non-stationaire overgangskansen voorspellen
# =========================

def voorspel_kansen(matrices, weegfactoren=None):
    if weegfactoren is None:
        weegfactoren = {
            2020: 1 / 1.01,
            2021: 1 / 0.99
        }

    jaren = sorted(matrices.keys())

    regressie_voorspellingen = {}
    smoothing_voorspellingen = {}

    geldige_overgangen = [
        (src, dst)
        for src in toestanden
        for dst in toestanden
        if src != 'uitstroom'
    ]

    for src, dst in geldige_overgangen:
        y = np.array([
            matrices[jaar].loc[src, dst] * weegfactoren.get(jaar, 1)
            for jaar in jaren
        ])

        if len(y) >= 3 and y.sum() > 0.01:
            X = np.arange(len(y)).reshape(-1, 1)

            try:
                regressie_voorspellingen[(src, dst)] = LinearRegression().fit(X, y).predict([[len(y)]])[0]
            except Exception:
                regressie_voorspellingen[(src, dst)] = 0

            try:
                trend_type = 'mul' if np.all(y > 0) else 'add'
                model = ExponentialSmoothing(y, trend=trend_type).fit(optimized=True)
                smoothing_voorspellingen[(src, dst)] = model.forecast(1)[0]
            except Exception:
                smoothing_voorspellingen[(src, dst)] = 0
        else:
            regressie_voorspellingen[(src, dst)] = 0
            smoothing_voorspellingen[(src, dst)] = 0

    return regressie_voorspellingen, smoothing_voorspellingen, jaren


# =========================
# Voorspelde overgangsmatrix maken
# =========================

def maak_voorspelde_matrix(voorspellingen):
    mat = pd.DataFrame(0.0, index=toestanden, columns=toestanden)

    for (src, dst), waarde in voorspellingen.items():
        mat.loc[src, dst] = max(0, waarde)

    mat.loc['uitstroom', :] = 0
    mat.loc['uitstroom', 'uitstroom'] = 1

    return mat.div(mat.sum(axis=1), axis=0).fillna(0)


# =========================
# Vectoren en echte aantallen
# =========================

def startvector_uit_huidige_status(df, jaar):
    vector = (
        df[df['Studiejaar'] == jaar]
        .groupby('huidige_status')['telling']
        .sum()
        .reindex(toestanden, fill_value=0)
        .astype(float)
    )

    # Uitstroom zit niet in de actieve startpopulatie
    vector.loc['uitstroom'] = 0

    return vector


def echte_aantallen_voor_foutmarge(df, jaar):
    jaar_df = df[df['Studiejaar'] == jaar]

    aantallen = (
        jaar_df
        .groupby('huidige_status')['telling']
        .sum()
        .reindex(toestanden, fill_value=0)
        .astype(float)
    )

    # Uitstroom komt uit volgende_status
    aantallen.loc['uitstroom'] = jaar_df[
        jaar_df['volgende_status'] == 'uitstroom'
    ]['telling'].sum()

    return aantallen


def nieuwe_instroom_uit_df(df, jaar):
    return (
        df[
            (df['Studiejaar'] == jaar) &
            (df['huidige_status'].isin(nieuwe_statussen))
        ]
        .groupby('huidige_status')['telling']
        .sum()
        .reindex(toestanden, fill_value=0)
        .astype(float)
    )


# =========================
# Voorspellen
# =========================

def voorspel_jaar(startvector, matrix, voorspeljaar, instroomvector):
    start = startvector.copy()
    start.loc['uitstroom'] = 0

    voorspeld = start.values @ matrix.values

    voorspeld = pd.Series(
        voorspeld,
        index=toestanden,
        name=voorspeljaar
    )

    # Nieuwe instroom wordt niet door de matrix voorspeld, maar apart ingevoerd
    for status in nieuwe_statussen:
        voorspeld.loc[status] = instroomvector.loc[status]

    return voorspeld


def bepaal_instroomvector(df, jaar, prognose_jaren, instroom_aanname_jaar):
    beschikbare_jaren = set(df['Studiejaar'].unique())

    if jaar in beschikbare_jaren and jaar not in prognose_jaren:
        return nieuwe_instroom_uit_df(df, jaar)

    return nieuwe_instroom_uit_df(df, instroom_aanname_jaar)


# =========================
# Foutmarges
# =========================

def vergelijk_voorspelling_met_echt(df, voorspeld, jaar):
    echt = echte_aantallen_voor_foutmarge(df, jaar)

    resultaat = pd.DataFrame({
        'echt': echt,
        'voorspeld': voorspeld,
        'verschil': voorspeld - echt,
        'absolute_fout': (voorspeld - echt).abs(),
        'foutpercentage': ((voorspeld - echt) / echt.replace(0, np.nan)) * 100
    })

    return resultaat.round(2)


# =========================
# Samenvattingen
# =========================

def totaal_nieuw(series):
    return series.loc[nieuwe_statussen].sum()


def totaal_doorstroom(series):
    return series.loc[doorstroom_statussen].sum()


def totaal_uitstroom(series):
    return series.loc['uitstroom']


def maak_samenvatting(voorspelling, df=None, jaar=None):
    voorspeld_samenvatting = {
        'nieuw': totaal_nieuw(voorspelling),
        'doorstroom': totaal_doorstroom(voorspelling),
        'uitstroom': totaal_uitstroom(voorspelling)
    }

    if df is None or jaar is None:
        return pd.DataFrame({'voorspeld': voorspeld_samenvatting}).round(0)

    echt = echte_aantallen_voor_foutmarge(df, jaar)

    resultaat = pd.DataFrame({
        'echt': {
            'nieuw': totaal_nieuw(echt),
            'doorstroom': totaal_doorstroom(echt),
            'uitstroom': totaal_uitstroom(echt)
        },
        'voorspeld': voorspeld_samenvatting
    })

    resultaat['verschil'] = resultaat['voorspeld'] - resultaat['echt']
    resultaat['absolute_fout'] = resultaat['verschil'].abs()
    resultaat['foutpercentage'] = (
        resultaat['verschil'] / resultaat['echt'].replace(0, np.nan)
    ) * 100

    return resultaat.round(2)


# =========================
# Model bouwen
# =========================

def bouw_modellen(df, train_jaren):
    matrices = bouw_jaar_matrices(df, jaren=train_jaren)

    regressie, smoothing, gebruikte_jaren = voorspel_kansen(matrices)

    model_matrices = {
        'regressie': maak_voorspelde_matrix(regressie),
        'exponential_smoothing': maak_voorspelde_matrix(smoothing),
        'stationair': stationaire_voorspelling(matrices)
    }

    return model_matrices, gebruikte_jaren


# =========================
# Complete run
# =========================

def draai_voorspelmodel(
    df,
    train_jaren,
    test_jaren,
    prognose_jaren,
    instroom_aanname_jaar,
    modellen_gebruiken=None
):
    if modellen_gebruiken is None:
        modellen_gebruiken = ['regressie', 'exponential_smoothing', 'stationair']

    valideer_input(df)

    train_jaren = list(train_jaren)
    alle_voorspeljaren = list(test_jaren) + list(prognose_jaren)
    startjaar = max(train_jaren)

    model_matrices, gebruikte_jaren = bouw_modellen(df, train_jaren)

    voorspellingen = {}
    foutmarges = {}
    samenvattingen = {}

    for modelnaam in modellen_gebruiken:
        matrix = model_matrices[modelnaam]
        vorige_vector = startvector_uit_huidige_status(df, startjaar)

        voorspellingen[modelnaam] = {}
        foutmarges[modelnaam] = {}
        samenvattingen[modelnaam] = {}

        for jaar in alle_voorspeljaren:
            instroomvector = bepaal_instroomvector(
                df=df,
                jaar=jaar,
                prognose_jaren=prognose_jaren,
                instroom_aanname_jaar=instroom_aanname_jaar
            )

            voorspelling = voorspel_jaar(
                startvector=vorige_vector,
                matrix=matrix,
                voorspeljaar=jaar,
                instroomvector=instroomvector
            )

            voorspellingen[modelnaam][jaar] = voorspelling

            if jaar in test_jaren:
                foutmarges[modelnaam][jaar] = vergelijk_voorspelling_met_echt(df, voorspelling, jaar)
                samenvattingen[modelnaam][jaar] = maak_samenvatting(voorspelling, df=df, jaar=jaar)
            else:
                samenvattingen[modelnaam][jaar] = maak_samenvatting(voorspelling)

            vorige_vector = voorspelling

    metadata = {
        'train_jaren': gebruikte_jaren,
        'startjaar': startjaar,
        'test_jaren': list(test_jaren),
        'prognose_jaren': list(prognose_jaren),
        'instroom_aanname_jaar': instroom_aanname_jaar
    }

    return voorspellingen, foutmarges, samenvattingen, model_matrices, metadata


# =========================
# Resultaten tonen
# =========================

def toon_resultaten(voorspellingen, foutmarges, samenvattingen, metadata):
    print("Jaren gebruikt voor training:", metadata['train_jaren'])
    print("Startjaar:", metadata['startjaar'])
    print("Testjaren:", metadata['test_jaren'])
    print("Prognosejaren:", metadata['prognose_jaren'])
    print("Instroomaanname voor prognosejaren:", metadata['instroom_aanname_jaar'])

    for modelnaam, jaren_dict in voorspellingen.items():
        print("\n" + "=" * 80)
        print(f"MODEL: {modelnaam}")
        print("=" * 80)

        for jaar, voorspelling in jaren_dict.items():
            print(f"\nVoorspelling {jaar} - {modelnaam}:")
            display(voorspelling.round(0))

            print(f"\nSamenvatting {jaar} - {modelnaam}:")
            display(samenvattingen[modelnaam][jaar])

            if jaar in foutmarges[modelnaam]:
                print(f"\nFoutmarge per status {jaar} - {modelnaam}:")
                display(foutmarges[modelnaam][jaar])


# =========================
# Uitvoeren
# df is al ingeladen
# =========================

voorspellingen, foutmarges, samenvattingen, model_matrices, metadata = draai_voorspelmodel(
    df=df,
    train_jaren=TRAIN_JAREN,
    test_jaren=TEST_JAREN,
    prognose_jaren=PROGNOSE_JAREN,
    instroom_aanname_jaar=INSTROOM_AANNAME_JAAR,
    modellen_gebruiken=MODELLEN_GEBRUIKEN
)

toon_resultaten(voorspellingen, foutmarges, samenvattingen, metadata)


# =========================
# Handig: losse resultaten ophalen
# Voorbeelden:
# voorspellingen['regressie'][2026]
# samenvattingen['stationair'][2025]
# foutmarges['exponential_smoothing'][2024]
# model_matrices['regressie']
# =========================

In [ ]:
# =========================
# Plotly grafieken overgangskansen
# Apart te draaien nadat df al is ingeladen
# =========================

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots


# =========================
# Configuratie plots
# =========================

PLOT_JAREN = range(2017, 2026)
MINIMUM_GEMIDDELDE_STUDENTEN = 100
AANTAL_KOLOMMEN = 2
HOOGTE_PER_RIJ = 320


# =========================
# Toestanden
# Alleen overgangen tussen deze toestanden worden meegenomen
# =========================

toestanden = [
    'Niveau1_jaar_1_nieuw', 'Niveau1_jaar_1', 'Niveau1_jaar_2', 'Niveau1_jaar_>2',
    'Niveau2_jaar_1_nieuw', 'Niveau2_jaar_1', 'Niveau2_jaar_2', 'Niveau2_jaar_>2',
    'Niveau3_jaar_1_nieuw', 'Niveau3_jaar_1', 'Niveau3_jaar_2', 'Niveau3_jaar_3', 'Niveau3_jaar_>3',
    'Niveau4_jaar_1_nieuw', 'Niveau4_jaar_1', 'Niveau4_jaar_2', 'Niveau4_jaar_3',
    'Niveau4_jaar_4', 'Niveau4_jaar_5', 'Niveau4_jaar_>5',
    'uitstroom'
]


# =========================
# Input controleren
# =========================

def controleer_plot_input(df):
    verplichte_kolommen = {
        'Studiejaar',
        'huidige_status',
        'volgende_status',
        'telling'
    }

    ontbrekend = verplichte_kolommen - set(df.columns)

    if ontbrekend:
        raise ValueError(f"Ontbrekende kolommen in df: {ontbrekend}")


# =========================
# Data voorbereiden
# =========================

def overgangskansen_per_jaar(
    df,
    jaren=None,
    minimum_gemiddelde_studenten=100,
    toestanden_lijst=None,
    toon_verwijderde_overgangen=True
):
    controleer_plot_input(df)

    if toestanden_lijst is None:
        toestanden_lijst = toestanden

    if jaren is None:
        jaren = sorted(df['Studiejaar'].dropna().unique())
    else:
        jaren = list(jaren)

    basis = df[df['Studiejaar'].isin(jaren)].copy()

    # Alleen geldige overgangen meenemen:
    # huidige_status en volgende_status moeten allebei in toestanden staan.
    ongeldig_masker = (
        ~basis['huidige_status'].isin(toestanden_lijst)
        |
        ~basis['volgende_status'].isin(toestanden_lijst)
    )

    if toon_verwijderde_overgangen and ongeldig_masker.sum() > 0:
        print(
            f"{ongeldig_masker.sum()} regels verwijderd omdat huidige_status of volgende_status niet in toestanden staat."
        )

        display(
            basis.loc[
                ongeldig_masker,
                ['huidige_status', 'volgende_status']
            ]
            .drop_duplicates()
            .sort_values(['huidige_status', 'volgende_status'])
        )

    basis = basis.loc[~ongeldig_masker].copy()

    # Specifieke correctie:
    # Uitstroom in 2025 niet meenemen in de plots,
    # maar 2025 wel behouden voor alle andere overgangen.
    uitstroom_2025_masker = (
        (basis['Studiejaar'] == 2025)
        & (basis['volgende_status'] == 'uitstroom')
    )

    if uitstroom_2025_masker.sum() > 0:
        print(
            f"{uitstroom_2025_masker.sum()} regels verwijderd: uitstroom 2025 wordt niet meegenomen in de plots."
        )

    basis = basis.loc[~uitstroom_2025_masker].copy()

    # Alleen geldige tellingen meenemen
    basis['telling'] = pd.to_numeric(basis['telling'], errors='coerce')
    basis = basis.dropna(subset=['telling'])
    basis = basis[basis['telling'] > 0].copy()

    overgang_counts = (
        basis
        .groupby(
            ['Studiejaar', 'huidige_status', 'volgende_status'],
            as_index=False
        )['telling']
        .sum()
    )

    totaal_per_huidige_status = (
        basis
        .groupby(
            ['Studiejaar', 'huidige_status'],
            as_index=False
        )['telling']
        .sum()
        .rename(columns={'telling': 'totaal_huidige_status'})
    )

    overgang_counts = overgang_counts.merge(
        totaal_per_huidige_status,
        on=['Studiejaar', 'huidige_status'],
        how='left'
    )

    overgang_counts['overgangskans'] = (
        overgang_counts['telling'] /
        overgang_counts['totaal_huidige_status']
    )

    # NaN, inf en -inf niet meenemen
    overgang_counts['overgangskans'] = overgang_counts['overgangskans'].replace(
        [np.inf, -np.inf],
        np.nan
    )

    overgang_counts = overgang_counts.dropna(
        subset=['overgangskans']
    )

    overgang_counts = overgang_counts[
        np.isfinite(overgang_counts['overgangskans'])
    ].copy()

    gemiddelde_studenten = (
        overgang_counts
        .groupby(
            ['huidige_status', 'volgende_status'],
            as_index=False
        )['telling']
        .mean()
        .rename(columns={'telling': 'gemiddelde_studenten_per_jaar'})
    )

    grote_overgangen = gemiddelde_studenten[
        gemiddelde_studenten['gemiddelde_studenten_per_jaar'] > minimum_gemiddelde_studenten
    ].copy()

    resultaat = overgang_counts.merge(
        grote_overgangen,
        on=['huidige_status', 'volgende_status'],
        how='inner'
    )

    resultaat['overgang'] = (
        resultaat['huidige_status'] + ' → ' + resultaat['volgende_status']
    )

    resultaat = resultaat.dropna(
        subset=[
            'Studiejaar',
            'huidige_status',
            'volgende_status',
            'telling',
            'totaal_huidige_status',
            'overgangskans',
            'gemiddelde_studenten_per_jaar',
            'overgang'
        ]
    )

    resultaat = resultaat[
        resultaat['huidige_status'].isin(toestanden_lijst)
        & resultaat['volgende_status'].isin(toestanden_lijst)
        & np.isfinite(resultaat['overgangskans'])
    ].copy()

    return resultaat.sort_values(['overgang', 'Studiejaar']).reset_index(drop=True)


# =========================
# Plot maken
# =========================

def plot_overgangskansen_subplots(
    df,
    jaren=None,
    minimum_gemiddelde_studenten=100,
    kolommen=2,
    hoogte_per_rij=320,
    toestanden_lijst=None,
    toon_verwijderde_overgangen=True
):
    plotdata = overgangskansen_per_jaar(
        df=df,
        jaren=jaren,
        minimum_gemiddelde_studenten=minimum_gemiddelde_studenten,
        toestanden_lijst=toestanden_lijst,
        toon_verwijderde_overgangen=toon_verwijderde_overgangen
    )

    overgangen = sorted(plotdata['overgang'].dropna().unique())

    if len(overgangen) == 0:
        print(
            'Geen geldige overgangen gevonden met gemiddeld meer dan',
            minimum_gemiddelde_studenten,
            'studenten per jaar.'
        )
        return None, plotdata

    rijen = int(np.ceil(len(overgangen) / kolommen))

    fig = make_subplots(
        rows=rijen,
        cols=kolommen,
        subplot_titles=overgangen,
        shared_xaxes=False,
        shared_yaxes=False
    )

    for i, overgang in enumerate(overgangen):
        rij = (i // kolommen) + 1
        kolom = (i % kolommen) + 1

        subset = plotdata[
            (plotdata['overgang'] == overgang)
            & (np.isfinite(plotdata['overgangskans']))
        ].copy()

        if subset.empty:
            continue

        fig.add_trace(
            go.Scatter(
                x=subset['Studiejaar'],
                y=subset['overgangskans'],
                mode='lines+markers',
                name=overgang,
                customdata=np.stack(
                    [
                        subset['telling'],
                        subset['totaal_huidige_status'],
                        subset['gemiddelde_studenten_per_jaar']
                    ],
                    axis=-1
                ),
                hovertemplate=(
                    '<b>Jaar: %{x}</b><br>'
                    'Overgangskans: %{y:.2%}<br>'
                    'Aantal overgang: %{customdata[0]:,.0f}<br>'
                    'Totaal huidige status: %{customdata[1]:,.0f}<br>'
                    'Gemiddeld aantal per jaar: %{customdata[2]:,.0f}'
                    '<extra></extra>'
                )
            ),
            row=rij,
            col=kolom
        )

        fig.update_yaxes(tickformat='.0%', row=rij, col=kolom)
        fig.update_xaxes(dtick=1, row=rij, col=kolom)

    fig.update_layout(
        title=(
            f'Overgangskansen per jaar voor geldige overgangen met gemiddeld > '
            f'{minimum_gemiddelde_studenten} studenten per jaar'
        ),
        height=max(500, rijen * hoogte_per_rij),
        showlegend=False,
        template='plotly_white'
    )

    return fig, plotdata


# =========================
# Uitvoeren
# =========================

fig_overgangen, plotdata_overgangen = plot_overgangskansen_subplots(
    df=df,
    jaren=PLOT_JAREN,
    minimum_gemiddelde_studenten=MINIMUM_GEMIDDELDE_STUDENTEN,
    kolommen=AANTAL_KOLOMMEN,
    hoogte_per_rij=HOOGTE_PER_RIJ,
    toestanden_lijst=toestanden,
    toon_verwijderde_overgangen=True
)

if fig_overgangen is not None:
    fig_overgangen.show()


# =========================
# Optioneel: data achter de grafieken bekijken
# =========================

# display(plotdata_overgangen)


# =========================
# Optioneel: aantal meegenomen overgangen bekijken
# =========================

print(
    'Aantal meegenomen overgangen:',
    plotdata_overgangen['overgang'].nunique()
)
